# 01 · Toei GTFS-Pathways, Ōedo Line

Step 1.2. Analysis logic lives in `importer/analysis/pathways.py` (unit-tested); this notebook runs it and shows the results.

- Step-free = walkway, moving walkway, elevator, fare gate, exit gate. Stairs and escalators are excluded. Fare gates count as passable because the data has no gate width.
- *Street entrance* = entrance node at level ≥ 0.
- Reachability is checked on the directed graph: `is_bidirectional = 0` edges are one-way.

In [1]:
import pandas as pd
from importer.analysis.pathways import *

pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_colwidth', 80)
feed = load_feed()
print({k: len(getattr(feed, k)) for k in ('stops', 'pathways', 'levels')})

{'stops': 2735, 'pathways': 2864, 'levels': 18}


## 1. What the feed contains

Columns present in `pathways.txt` and `stops.txt`, and which optional pathway fields are missing.

In [2]:
print('pathways.txt columns:', list(feed.pathways.columns))
missing = ['traversal_time','stair_count','max_slope','min_width','signposted_as','reversed_signposted_as']
print('GTFS optional fields absent:', [m for m in missing if m not in feed.pathways.columns])
print('stops with wheelchair_boarding set:', feed.stops.wheelchair_boarding.notna().sum(), 'of', len(feed.stops))
print('levels.txt:', feed.levels.level_id.tolist())
feed.stops.location_type.value_counts().rename(index={'0':'platform/stop','1':'station','2':'entrance','3':'generic node','4':'boarding area'})

pathways.txt columns: ['pathway_id', 'from_stop_id', 'to_stop_id', 'pathway_mode', 'is_bidirectional', 'length']
GTFS optional fields absent: ['traversal_time', 'stair_count', 'max_slope', 'min_width', 'signposted_as', 'reversed_signposted_as']
stops with wheelchair_boarding set: 0 of 2735
levels.txt: ['1', '0', '-0.2', '-0.5', '-0.7', '-1', '-1.5', '-2', '-2.5', '-3', '-3.5', '-4', '-4.5', '-5', '-5.5', '-6', '-6.5', '-7']


location_type
generic node     2194
boarding area     299
platform/stop     163
entrance           67
station            12
Name: count, dtype: int64

## 2. Stations with in-station Pathways

Only these stations have in-station nodes.

In [3]:
stations = tier2_stations(feed)
stations

,stop_id,stop_name,stop_code
0,402,新宿西口,E-01
1,403,東新宿,E-02
2,410,上野御徒町,E-09
3,421,大門,E-20
4,422,赤羽橋,E-21
5,423,麻布十番,E-22
6,424,六本木,E-23
7,425,青山一丁目,E-24
8,426,国立競技場,E-25
9,427,代々木,E-26


## 3. Summary per station

In [4]:
table = summary_table(feed)
table.drop(columns='issues')

,station,levels,nodes,pathways,walk,stairs,escalator,elevator,gate,elevator shafts,entrances (street / step-free to platform),platforms step-free from street,platforms step-free from any entrance,step-free back to street
0,新宿西口 (E-01),1 to -4,212,238,170,37,18,11,2,3,5 (3 / 2),0/2,2/2,0/2
1,東新宿 (E-02),0 to -3,100,107,78,16,6,6,1,2,4 (4 / 1),2/2,2/2,2/2
2,上野御徒町 (E-09),0 to -2,240,257,187,46,11,11,2,3,10 (8 / 1),2/2,2/2,2/2
3,大門 (E-20),0 to -5,329,364,265,56,25,17,1,6,13 (13 / 3),2/2,2/2,2/2
4,赤羽橋 (E-21),0 to -2,124,134,100,19,8,6,1,2,3 (3 / 1),2/2,2/2,2/2
5,麻布十番 (E-22),0 to -6,203,224,162,34,15,12,1,4,5 (5 / 1),2/2,2/2,2/2
6,六本木 (E-23),0 to -7,252,280,204,43,22,9,2,3,4 (3 / 1),2/2,2/2,2/2
7,青山一丁目 (E-24),0 to -5,253,273,207,36,11,17,2,7,6 (5 / 2),2/2,2/2,2/2
8,国立競技場 (E-25),0 to -3,285,314,227,52,21,12,2,4,6 (6 / 1),2/2,2/2,2/2
9,代々木 (E-26),0 to -3,130,143,107,21,8,6,1,2,2 (1 / 1),2/2,2/2,2/2


## 4. Platform reachability detail

Per platform: boarding areas, and step-free reachability in both directions. `any mode` is the same check allowing stairs and escalators, to separate “missing elevator data” from “disconnected graph”.

In [5]:
rows = []
results = {}
for st in stations.itertuples():
    r = analyse_station(feed, st.stop_id)
    results[st.stop_id] = r
    for p in r['platforms']:
        rows.append({'station': st.stop_name, **p})
pd.DataFrame(rows)

,station,platform,boarding_areas,in_from_street_free,in_from_any_entrance_free,out_to_street_free,in_from_street_any_mode
0,新宿西口,402P1,6,False,True,False,True
1,新宿西口,402P2,6,False,True,False,True
2,東新宿,403P1,11,True,True,True,True
3,東新宿,403P2,11,True,True,True,True
4,上野御徒町,410P1,11,True,True,True,True
5,上野御徒町,410P2,11,True,True,True,True
6,大門,421P3,15,True,True,True,True
7,大門,421P4,15,True,True,True,True
8,赤羽橋,422P1,11,True,True,True,True
9,赤羽橋,422P2,11,True,True,True,True


## 5. Stations where a platform is not step-free from a street-level entrance

Check whether the step-free component around the platforms touches any entrance.

In [6]:
bad = [(st.stop_name, st.stop_id) for st in stations.itertuples() if not all(p['in_from_street_free'] for p in results[st.stop_id]['platforms'])]
for name, sid in bad:
    nodes = station_nodes(feed, sid)
    edges = station_edges(feed, set(nodes.stop_id))
    g = build_graph(edges, True)
    boarding = set(nodes[nodes.location_type == '4'].stop_id)
    comp = set()
    for b in boarding:
        if b in g:
            comp |= nx.descendants(g, b) | nx.ancestors(g, b) | {b}
    ent = nodes[nodes.location_type == '2'][['stop_id', 'stop_name', 'level_id']].assign(in_platform_step_free_component=lambda d: d.stop_id.isin(comp))
    levels = sorted({float(l) for l in nodes[nodes.stop_id.isin(comp)].level_id.dropna()}, reverse=True)
    print(name, sid, '| step-free component levels:', levels)
    print(ent.to_string(index=False), '\n')

新宿西口 402 | step-free component levels: [-0.5, -1.0, -3.0, -4.0]
 stop_id stop_name level_id  in_platform_step_free_component
402N0000        D5        0                            False
402N0001        D4        0                            False
402N0002        D3        0                            False
402N0017        D2     -0.5                             True
402N0020        D1       -1                             True 

新宿 428 | step-free component levels: [-1.0, -2.0, -3.0, -6.0]
 stop_id stop_name level_id  in_platform_step_free_component
428N0023         3        0                            False
428N0193        A1        0                            False
428N0195         4        0                            False 



## 6. Data errors and oddities

Counts per check, per station. A blank cell means zero. IDs are in `results[station]['issues']`.

In [7]:
names = stations.set_index('stop_id').stop_name
iss = pd.DataFrame({names[sid]: {k: len(v) for k, v in r['issues'].items()} for sid, r in results.items()}).T
iss.replace(0, '')

,node_without_level,level_not_in_levels_txt,isolated_node,self_loop,duplicate_edge,zero_or_missing_length,level_change_without_vertical_mode,vertical_mode_same_level,edge_leaves_station,node_far_from_station,length_vs_distance_mismatch
新宿西口,,,,,,,1,26,,,
東新宿,,,,,,,,12,,,
上野御徒町,,,,,,,,34,,,
大門,,,,,6,,,28,,,
赤羽橋,,,,,,,,17,,,
麻布十番,,,,,,,,20,,,
六本木,,,,,,,,27,,,
青山一丁目,,,,,,,2,31,,,
国立競技場,,,,,,,,44,,,
代々木,,,,,,,,17,,,


Examples of each issue type (first 3 IDs):

In [8]:
seen = {}
for sid, r in results.items():
    for k, v in r['issues'].items():
        if v and k not in seen:
            seen[k] = (names[sid], v[:3])
seen

{'level_change_without_vertical_mode': ('新宿西口', ['402L0005']),
 'vertical_mode_same_level': ('新宿西口', ['402L0009', '402L0014', '402L0018']),
 'duplicate_edge': ('大門', ['421L0340', '421L0341', '421L0342'])}

### Vertical modes on the same level

Stairs, escalators and elevators between two nodes with the same `level_id`. Lengths show whether these are hall-to-cab hops and half-flights (expected with 0.5 level resolution) or errors.

In [9]:
rows = []
for sid, r in results.items():
    ids = set(r['issues']['vertical_mode_same_level'])
    e = feed.pathways[feed.pathways.pathway_id.isin(ids)].copy()
    e['length'] = e.length.astype(float)
    rows.append(e)
same = pd.concat(rows)
same.groupby(same.pathway_mode.map(MODE_NAMES)).length.describe()

,count,mean,std,min,25%,50%,75%,max
pathway_mode,,,,,,,,
elevator,85.0,1.498824,0.504148,0.5,1.2,1.4,1.70,3.5
escalator,31.0,9.190323,4.173836,3.2,5.1,9.6,12.75,15.9
stairs,185.0,4.389189,1.509208,0.7,3.6,4.4,5.10,11.7
